# News Topic Classifier using BERT

## Problem Satement and Objectives

Manually categorizing news articles into predefined topics such as World, Sports, Business, and Science & Technology is time-consuming, error-prone, and not scalable. Traditional machine learning approaches rely heavily on manual feature engineering and often fail to capture the contextual meaning of text, leading to suboptimal classification performance, there is a need for an automated, intelligent, and context-aware system that can accurately classify news headlines into relevant topic categories using advanced Natural Language Processing (NLP) techniques

### Installing Required Libraries

In [1]:
pip install transformers datasets torch scikit-learn streamlit evaluate


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 71.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 90.0 MB/s eta 0:00:00


### Importing Libraries

In [2]:
import torch
import numpy as np
from datasets import load_dataset
from transformers import (BertTokenizerFast, BertForSequenceClassification, Trainer, TrainingArguments)
from sklearn.metrics import accuracy_score, f1_score


### Loading AG News Dataset

In [3]:
dataset = load_dataset("ag_news")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/18.6M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

In [4]:
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 7600
    })
})


### Tokenization & Preprocessing

In [5]:
tokenizer = BertTokenizerFast.from_pretrained("bert-base-uncased")

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

tokenized_datasets = dataset.map(tokenize_function, batched=True)


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

Map:   0%|          | 0/120000 [00:00<?, ? examples/s]

Map:   0%|          | 0/7600 [00:00<?, ? examples/s]

### Prepare Dataset for Training

In [6]:
tokenized_datasets = tokenized_datasets.remove_columns(["text"])
tokenized_datasets = tokenized_datasets.rename_column("label", "labels")
tokenized_datasets.set_format("torch")


### Split

In [7]:
train_dataset = tokenized_datasets["train"]
test_dataset = tokenized_datasets["test"]


### Loading Bert Model

In [8]:
model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=4
)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


### Defining Evaluation Metrics

In [9]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)

    acc = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions, average="weighted")

    return {
        "accuracy": acc,
        "f1": f1
    }

### Training Configuration

In [12]:
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_dir="./logs"
)



### Train the Model

In [13]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

trainer.train()


/tmp/ipython-input-726133461.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Step,Training Loss
500,0.301000
1000,0.257700
1500,0.234200
2000,0.221400
2500,0.240000
3000,0.239000
3500,0.213400
4000,0.208100
4500,0.203200
5000,0.196800


TrainOutput(global_step=15000, training_loss=0.17363256581624348, metrics={'train_runtime': 6772.0868, 'train_samples_per_second': 35.44, 'train_steps_per_second': 2.215, 'total_flos': 1.578694680576e+16, 'train_loss': 0.17363256581624348, 'epoch': 2.0})

### Saving the Model

In [14]:
model.save_pretrained("news_bert_model")
tokenizer.save_pretrained("news_bert_model")


('news_bert_model/tokenizer_config.json',
 'news_bert_model/special_tokens_map.json',
 'news_bert_model/vocab.txt',
 'news_bert_model/added_tokens.json',
 'news_bert_model/tokenizer.json')

## Final Summary and Insights

This project successfully demonstrates the application of transformer-based transfer learning for news topic classification. By fine-tuning the bert-base-uncased model on the AG News dataset, the system achieved high classification performance without the need for manual feature extraction.

The use of BERTs bidirectional attention mechanism allowed the model to capture contextual relationships between words more effectively than traditional machine learning models. Evaluation results showed strong accuracy and F1-score values, indicating that the model generalizes well across different news categories.